# PEGASE 1354 with two embedded HVDC grids

The PEGASE 1354-bus model of part of the European transmission grid, with two HVDC grids embedded in it. A realistically sized case for convergence and run time.

| | |
|---|---|
| AC buses | 1354 (1751 lines, 240 transformers, 259 generators) |
| DC buses | 5, in 2 DC grids |
| Converters | 5: 2 × VdcQ (holds its DC voltage, fixed Q), 3 × PQ (fixed P and Q) |
| Case file | `validation/cases/pegase_hvdc.yaml` |

## What this notebook does

1. **Load** the case file into a pandapower net and classify the nodes. `LOSSY = True` adds the converter
   losses a + b·|I| + c·|I|² (coefficients of the paper, section IV-B).
2. **Build Y**: the AC admittance matrix (lines, transformer taps and shifts, shunts) and the DC conductance
   matrix.
3. **Solve** the AC grids, the DC grids and the converters together in one Newton–Raphson, and print the
   iterations, the largest mismatch and the run time.
4. **Results**: per node the voltage E = E_re + j·E_im (p.u.) and the injected power P, Q (p.u., positive into
   the grid); per converter its AC and DC power, losses and current.

## Try it

- Change a converter in the `converter` table of the case file, for example its `mode`, `p_mw`, `q_mvar` or
  `vdc_pu`, then *Run All*. Modes: PQ, VdcQ, PV, VdcV, GF (grid-forming) and droop; the columns each mode uses are
  listed at the top of the case file.
- Set `LOSSY = True` in the first cell to include converter losses.

## Running this notebook

Use the kernel **Python 3.12 (unified-acdc-powerflow)** (VS Code: *Select Kernel* → *Jupyter Kernel…*). To set it
up once, in the repository folder:

```bash
python3.12 -m venv .venv
.venv/bin/pip install -e ".[dev]"
.venv/bin/python -m ipykernel install --user --name unified-acdc-powerflow --display-name "Python 3.12 (unified-acdc-powerflow)"
```


In [1]:
import logging, warnings
logging.getLogger("pandapower").setLevel(logging.ERROR)
warnings.filterwarnings("ignore")
import pandas as pd
pd.set_option("display.max_rows", 2000)
from unified_acdc_powerflow import build_model, build_ybus, solve
from unified_acdc_powerflow.cases import load_case

# 1) load the case and classify the nodes
CASE = "pegase_hvdc"
LOSSY = False          # True: converter losses a + b|I| + c|I|^2 (coefficients of the paper)
net = load_case(CASE, lossy=LOSSY)   # pandapower net from validation/cases/pegase_hvdc.yaml
model = build_model(net)
print(f"{model.n_ac} AC nodes, {model.n_dc} DC nodes, {model.n_conv} converters")
net.vsc[["name", "bus", "bus_dc", "mode", "p_ac_mw", "q_ac_mvar", "vm_dc_pu", "p_dc_ref_mw", "droop_k"]]


1354 AC nodes, 5 DC nodes, 5 converters


,name,bus,bus_dc,mode,p_ac_mw,q_ac_mvar,vm_dc_pu,p_dc_ref_mw,droop_k
0,IC 2072-9261,2072,9261,VdcQ,0.0,-10.0,1.0,0.0,0.0
1,IC 8195-9262,8195,9262,PQ,10.0,-10.0,1.0,0.0,0.0
2,IC 6246-9263,6246,9263,VdcQ,0.0,-10.0,1.0,0.0,0.0
3,IC 7282-9264,7282,9264,PQ,-10.0,-10.0,1.0,0.0,0.0
4,IC 352-9265,352,9265,PQ,-10.0,-10.0,1.0,0.0,0.0


In [2]:
# 2) build Y (AC: pandapower Ybus with taps and shunts; DC: conductance matrix G)
build_ybus(model)
print("Y (AC):", model.Y.shape, "nnz", model.Y.nnz)
print("G (DC):", model.G.shape, "nnz", model.G.nnz)


Y (AC): (1354, 1354) nnz 4774
G (DC): (5, 5) nnz 13


In [3]:
# 3) Newton-Raphson
res = solve(model)
print(res.message, "| iterations:", res.iterations, "| max mismatch:", f"{res.max_mismatch:.2e}",
      "| time:", f"{res.solve_time_s * 1e3:.1f} ms")


converged | iterations: 7 | max mismatch: 3.11e-12 | time: 1408.9 ms


In [4]:
# 4) results: per node E (real, imag) and injected P, Q in p.u.; then per converter
display(res.table())
res.converter_table()


,side,bus,name,type,E_re,E_im,P_pu,Q_pu
0,ac,3,3,PQ,0.971144,-0.319170,-1.510000e-01,-4.880000e-02
1,ac,4,4,PQ,1.002197,-0.305644,-1.714100e-01,-2.340000e-02
2,ac,10,10,PQ,1.022246,-0.345358,-1.340000e-01,-2.470000e-02
3,ac,21,21,PQ,1.015061,-0.338419,-1.612000e-01,-3.930000e-02
4,ac,22,22,PQ,1.017501,-0.332868,2.230897e-15,3.128429e-14
5,ac,26,26,PQ,0.991684,-0.295836,-1.673000e-01,-5.480000e-02
6,ac,29,29,PQ,0.951129,-0.300441,-1.529000e-01,-4.940000e-02
7,ac,44,44,PQ,1.023492,-0.341720,-3.669634e-14,-5.241211e-14
8,ac,53,53,PQ,1.017675,-0.313685,-4.865000e-02,-1.610000e-02
9,ac,58,58,PQ,0.939945,-0.297707,-1.782637e-15,-2.883019e-14


,vsc,name,mode,P_ac_pu,Q_ac_pu,P_dc_pu,loss_pu,I_pu
0,0,IC 2072-9261,VdcQ,-0.01,-0.01,0.01,0.0,0.013282
1,1,IC 8195-9262,PQ,0.01,-0.01,-0.01,0.0,0.013868
2,2,IC 6246-9263,VdcQ,0.02,-0.01,-0.02,0.0,0.021555
3,3,IC 7282-9264,PQ,-0.01,-0.01,0.01,0.0,0.013670
4,4,IC 352-9265,PQ,-0.01,-0.01,0.01,0.0,0.013400
